# Week 5, Lab 02: Cordwell Support-Doc Retrieval

Cordwell Home and Hardware has a support knowledge base: short FAQ entries,
medium troubleshooting guides, and long installation manuals. Support engineers
keep saying the same thing. The right document exists. It does not come back.

Your job is to build the index, measure how bad it is, and improve it with
evidence rather than intuition.

## What you will have built by the end

1. A chunking pipeline with structured, delete-friendly IDs
2. A vector index with a metadata schema you can actually filter on
3. An evaluation harness that turns chunking from a guess into a decision
4. A coverage audit that catches a class of bug no exception will ever show you

## Where this stops

This lab ends at "the right chunks came back". Turning those chunks into a
grounded answer is the next module. No text generation happens here.

## Ground rules

Every number you produce is real. If your figures differ from the ones written
in the README, that is data, not a mistake, and it is worth a minute of thought
about why.

In [ ]:
%pip install -r requirements.txt

## Setup

Run this cell first. It reports which backend you are on and fails loudly if the
selection is inconsistent, rather than quietly switching to something else.

In [ ]:
import os, sys, time
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt

import lab_support as L
from lab_support import (
    LABELED_QUERIES,
    load_corpus, length_report, plot_length_distribution, plot_metric_comparison,
    tokenize, count_tokens, split_paragraphs,
    to_epoch, content_hash,
    get_embedder, get_index,
    recall_at_k, reciprocal_rank, evaluate,
    check, check_summary, reset_checks, pick_device, step,
)

reset_checks()

K = 5                # we report recall@5 and MRR throughout
EMBED_DIM = 96       # only used by the offline LSA embedder

print(f"python           {sys.version.split()[0]}")
print(f"torch device     {pick_device()}")
print(f"LAB_BACKEND      {L.BACKEND}")
print(f"EMBED_BACKEND    {L.EMBEDDER}")
print(f"index / namespace {L.INDEX_NAME} / {L.NAMESPACE}")

In [ ]:
docs = load_corpus()
print(f"loaded {len(docs)} documents")

# The embedding model is fit ONCE, on paragraphs of the raw corpus, and then
# frozen. This matters more than it looks. A real embedding model was trained
# months ago by someone else and does not change when you change your chunk
# size. If you refit the embedder for every chunking configuration you are no
# longer measuring chunking, you are measuring chunking plus a different model,
# and the comparison means nothing.
background = [
    p for d in docs for p in split_paragraphs(d["text"]) if count_tokens(p) >= 8
]
print(f"embedder fit corpus: {len(background)} paragraphs")

t0 = time.time()
embedder = get_embedder("primary", dim=EMBED_DIM, background=background)
print(f"embedder: {embedder.name}  dimension={embedder.dimension}  "
      f"({time.time()-t0:.1f}s)")

# Shared state, initialised up front so that a cold Run All never dies on a
# NameError in a later cell. You will fill these in as you go.
results = {}          # label -> evaluate() output
indexes = {}          # label -> (index, records)
baseline_records, baseline_index, baseline_search = [], None, None
best_records, best_index = [], None

check("setup: corpus loaded", len(docs) > 150, f"{len(docs)} documents")
check("setup: embedder ready", embedder.dimension > 0, f"dim={embedder.dimension}")
check("setup: labeled query set", len(LABELED_QUERIES) == 30,
      f"{len(LABELED_QUERIES)} queries")

---

# Part A. Look at the data before you pick a chunk size

You cannot choose a chunk size without looking at your corpus. This is the step
teams skip, and skipping it is how you end up with a number someone read in a
blog post.

Run the provided cell, then write one function.

In [ ]:
stats = length_report(docs)
print(f"{'doc_type':18s} {'n':>5s} {'min':>7s} {'p50':>8s} {'p90':>8s} {'max':>7s}")
for doc_type, s in stats.items():
    print(f"{doc_type:18s} {s['count']:5d} {s['min']:7d} {s['p50']:8.1f} "
          f"{s['p90']:8.1f} {s['max']:7d}")

fig, ax = plot_length_distribution(docs)
plt.show()

### Task A1

Write `fraction_under(docs, size)`. It returns the fraction of documents in the
corpus whose token count is less than or equal to `size`.

This looks trivial. It is the single most load-bearing number in the lab, and
you will use it again in Part D to explain a result that otherwise looks like
magic.

**Worked target output.** With the real corpus, calling it looks exactly like
this:

```
fraction_under(docs, 128)   ->  0.9307
fraction_under(docs, 512)   ->  0.9703
fraction_under(docs, 1024)  ->  0.9703
fraction_under(docs, 4096)  ->  1.0000
```

Use `count_tokens(text)` from `lab_support`, which is already imported.

In [ ]:
def fraction_under(docs, size):
    """
    Fraction of documents whose token count is <= size.

    Args:
        docs: list of document dicts, each with a "text" key
        size: chunk size in word tokens

    Returns:
        float between 0.0 and 1.0
    """
    if not docs:
        return 0.0
    n_under = sum(1 for d in docs if count_tokens(d["text"]) <= size)
    return n_under / len(docs)

In [ ]:
for size in (128, 512, 1024, 4096):
    try:
        print(f"fraction_under(docs, {size:5d})  ->  {fraction_under(docs, size):.4f}")
    except NotImplementedError:
        print(f"fraction_under(docs, {size:5d})  ->  not implemented yet")

check("A1: fraction_under(128)",
      lambda: abs(fraction_under(docs, 128) - 0.9307) < 0.005)
check("A1: fraction_under(512)",
      lambda: abs(fraction_under(docs, 512) - 0.9703) < 0.005)
check("A1: fraction_under is monotone",
      lambda: fraction_under(docs, 128) <= fraction_under(docs, 512) <= fraction_under(docs, 4096))
check("A1: fraction_under(4096) covers everything",
      lambda: abs(fraction_under(docs, 4096) - 1.0) < 1e-9)

**Stop and read your own output.** Roughly 97 percent of this corpus is shorter
than 512 tokens, and the remaining 3 percent is ten to thirty times longer than
everything else. That is not a distribution with a middle. It is two populations
wearing a trench coat.

Hold that thought. It is the whole lab.

---

# Part B. Build the baseline index

Baseline configuration, straight off the slides: fixed size chunks of 1024
tokens with no overlap. We are not defending this choice. We are measuring it so
that later changes have something to be better than.

Three tasks: the chunker, the metadata, and the chunking pass over the corpus.

### Task B1

Write `chunk_fixed(tokens, size, overlap)`.

Fixed size chunks with overlap, over a list of tokens. Two properties matter:

- A non-empty input must always produce at least one chunk.
- `overlap >= size` is a caller bug that would otherwise loop forever. Guard it.

**Worked target output.**

```
chunk_fixed(list(range(10)), size=4, overlap=0)
  -> [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]]

chunk_fixed(list(range(10)), size=4, overlap=1)
  -> [[0, 1, 2, 3], [3, 4, 5, 6], [6, 7, 8, 9], [9]]

chunk_fixed([], size=4, overlap=0)
  -> []

chunk_fixed([1, 2], size=99, overlap=0)
  -> [[1, 2]]
```

In [ ]:
def chunk_fixed(tokens, size=1024, overlap=0):
    """
    Fixed size chunks with overlap.

    Args:
        tokens:  list of tokens
        size:    maximum tokens per chunk
        overlap: tokens repeated at each boundary

    Returns:
        list of token lists. Empty input returns []. Non-empty input always
        returns at least one chunk.
    """
    if not tokens:
        return []
    # Guard: overlap >= size would make step <= 0 and never advance.
    step = max(1, size - overlap)
    return [tokens[i:i + size] for i in range(0, len(tokens), step)]

In [ ]:
check("B1: splits evenly",
      lambda: chunk_fixed(list(range(10)), 4, 0) == [[0,1,2,3],[4,5,6,7],[8,9]])
check("B1: overlap repeats boundary tokens",
      lambda: chunk_fixed(list(range(10)), 4, 1)[1][0] == 3)
check("B1: empty input yields no chunks",
      lambda: chunk_fixed([], 4, 0) == [])
check("B1: short input yields exactly one chunk",
      lambda: chunk_fixed([1, 2], 99, 0) == [[1, 2]])
check("B1: overlap >= size does not hang",
      lambda: len(chunk_fixed(list(range(10)), 4, 9)) == 10)

### Task B2

Write `build_metadata(doc, chunk_index, total_chunks, chunk_text)`.

Two things here are load-bearing and both are easy to get wrong.

**Timestamps must be epoch integers.** Pinecone comparison operators such as
`$gte` need numeric operands. A string date does not raise. It simply never
matches, which is the worst failure mode available: the query looks like it
worked. Use `to_epoch(iso_string)`.

**The chunk ID is your only delete handle.** Use `{document_id}#{chunk_index}`.
Deleting a document later means listing IDs by prefix and deleting them. UUID
chunk IDs with no tracking make deletion an archaeology project.

Keep metadata lean. It is stored per vector and it counts against storage and
query latency. Store what you filter on or display, and keep the full document
elsewhere.

**Worked target output.** For the first chunk of `ts_thermostat_wifi`:

```python
{'chunk_id': 'ts_thermostat_wifi#0',
 'document_id': 'ts_thermostat_wifi',
 'source': 'troubleshoot_thermostat_wifi.md',
 'doc_type': 'troubleshooting',
 'product_line': 'climate',
 'created_at': 1755165600,
 'updated_at': 1779196320,
 'is_active': True,
 'chunk_index': 0,
 'total_chunks': 1,
 'text': 'Thermostat drops off the network after a power outage ...'}
```

Note `created_at` and `updated_at` are `int`, not `str`. Truncate `text` to 1200
characters.

In [ ]:
def build_metadata(doc, chunk_index, total_chunks, chunk_text):
    """
    Build the metadata payload for one chunk.

    Required keys: chunk_id, document_id, source, doc_type, product_line,
    created_at, updated_at, is_active, chunk_index, total_chunks, text

    created_at and updated_at must be epoch integers.
    text must be truncated to 1200 characters.
    """
    return {
        # {document_id}#{chunk_index} is what makes prefix deletion possible.
        "chunk_id": f"{doc['document_id']}#{chunk_index}",
        "document_id": doc["document_id"],
        "source": doc["source"],
        "doc_type": doc["doc_type"],
        "product_line": doc["product_line"],
        # Epoch integers, not ISO strings. $gte on a string silently matches
        # nothing and raises no error.
        "created_at": to_epoch(doc["created_at"]),
        "updated_at": to_epoch(doc["updated_at"]),
        "is_active": doc["is_active"],
        "chunk_index": chunk_index,
        "total_chunks": total_chunks,
        # Carried so results are displayable without a second lookup. Capped
        # because metadata size costs storage and query latency.
        "text": chunk_text[:1200],
    }

In [ ]:
_probe = [d for d in docs if d["document_id"] == "ts_thermostat_wifi"][0]
try:
    _meta = build_metadata(_probe, 0, 1, _probe["text"])
    pprint(_meta)
except NotImplementedError:
    _meta = None
    print("build_metadata not implemented yet")

check("B2: chunk_id uses {document_id}#{index}",
      lambda: build_metadata(_probe, 5, 12, "x")["chunk_id"] == "ts_thermostat_wifi#5")
check("B2: created_at is an int, not a string",
      lambda: isinstance(build_metadata(_probe, 0, 1, "x")["created_at"], int))
check("B2: updated_at is an int, not a string",
      lambda: isinstance(build_metadata(_probe, 0, 1, "x")["updated_at"], int))
check("B2: updated_at is later than created_at",
      lambda: (build_metadata(_probe, 0, 1, "x")["updated_at"]
               > build_metadata(_probe, 0, 1, "x")["created_at"]))
check("B2: text is capped at 1200 characters",
      lambda: len(build_metadata(_probe, 0, 1, "z" * 5000)["text"]) == 1200)
check("B2: is_active carried through",
      lambda: build_metadata(_probe, 0, 1, "x")["is_active"] is True)

### Task B3

Write `chunk_corpus(docs, strategy, **kwargs)`.

This is the pass that turns documents into indexable records. It dispatches on
`strategy` so that later parts can reuse it without a rewrite.

Supported strategies and what each receives:

| strategy | operates on | signature |
|---|---|---|
| `"fixed"` | token list | `chunk_fixed(tokens, size=..., overlap=...)` |
| `"sliding_v1"` | token list | `chunk_sliding_v1(tokens, size=..., stride=...)` |
| `"sliding_v2"` | token list | `chunk_sliding_v2(tokens, size=..., stride=...)` |
| `"paragraph"` | raw text | `chunk_paragraphs(text, size=..., overlap=...)` |

The token based strategies need `tokenize(doc["text"])` going in and
`" ".join(chunk)` coming out. The paragraph strategy takes and returns text
directly.

`chunk_sliding_v1`, `chunk_sliding_v2`, and `chunk_paragraphs` do not exist yet.
Dispatch to them anyway. Python resolves the name at call time, so the branches
you are not using yet will not fail.

**Worked target output.** One record looks like this:

```python
{'id': 'ts_thermostat_wifi#0',
 'text': 'Thermostat drops off the network after a power outage ...',
 'metadata': {...the dict from Task B2...}}
```

And over the whole corpus at the baseline setting:

```
chunk_corpus(docs, "fixed", size=1024, overlap=0)  ->  218 records
```

In [ ]:
def chunk_corpus(docs, strategy, **kwargs):
    """
    Chunk every document into indexable records.

    Returns a list of dicts, each with keys: id, text, metadata.
    The id must equal metadata["chunk_id"].

    total_chunks in the metadata is the number of chunks produced for THAT
    document, so it can only be computed after that document is chunked.
    """
    records = []
    for doc in docs:
        if strategy == "fixed":
            pieces = [" ".join(c) for c in chunk_fixed(tokenize(doc["text"]), **kwargs)]
        elif strategy == "sliding_v1":
            pieces = [" ".join(c) for c in chunk_sliding_v1(tokenize(doc["text"]), **kwargs)]
        elif strategy == "sliding_v2":
            pieces = [" ".join(c) for c in chunk_sliding_v2(tokenize(doc["text"]), **kwargs)]
        elif strategy == "paragraph":
            pieces = chunk_paragraphs(doc["text"], **kwargs)
        else:
            raise ValueError(f"unknown strategy: {strategy}")

        # total_chunks is per document, so it is only knowable now.
        total = len(pieces)
        for i, piece in enumerate(pieces):
            meta = build_metadata(doc, i, total, piece)
            records.append({"id": meta["chunk_id"], "text": piece, "metadata": meta})
    return records

In [ ]:
with step("Part B: chunk the corpus"):
    baseline_records = chunk_corpus(docs, "fixed", size=1024, overlap=0)
    print(f"baseline records: {len(baseline_records)}")
    pprint({k: (v if k != "text" else v[:70] + " ...")
            for k, v in baseline_records[0].items() if k != "metadata"})

check("B3: baseline produces 218 records",
      lambda: len(chunk_corpus(docs, "fixed", size=1024, overlap=0)) == 218,
      "expected 218 at size=1024 overlap=0")
check("B3: record id matches metadata chunk_id",
      lambda: bool(baseline_records) and
              all(r["id"] == r["metadata"]["chunk_id"] for r in baseline_records))
check("B3: chunk ids are unique",
      lambda: bool(baseline_records) and
              len({r["id"] for r in baseline_records}) == len(baseline_records))
check("B3: total_chunks is per document",
      lambda: bool(baseline_records) and
              all(r["metadata"]["total_chunks"] ==
                  sum(1 for x in baseline_records
                      if x["metadata"]["document_id"] == r["metadata"]["document_id"])
                  for r in baseline_records[:40]))

### Build the index

`build_index` is provided. Read it once. The only backend-specific line is
`get_index`. Everything after that point is identical whether you are talking to
an in-memory numpy store, the Pinecone emulator in Docker, or Pinecone
serverless in production.

To run pinecone locally, execute the following steps:

* On the Mac, execute ```brew install colima```
* Download **pinecone-local-latest.tar** from the **images** folder stored in https://gamuttechnologysvcs-my.sharepoint.com/:f:/p/asanders/IgD_SIVCz8YJQYh7BL3DUy4ZAVgU8-9SO8Lo3boIy-wwV8g?e=D4X53b
* From the location where **pinecone-local-latest.tar** has been stored, run `docker load -i pinecone-local-latest.tar` to load the image to local cache
* Run the container:

```bash
docker run --rm -d \
  --name pinecone-local \
  -e PORT=5080 \
  -e PINECONE_HOST=localhost \
  -p 5080-5090:5080-5090 \
  ghcr.io/pinecone-io/pinecone-local:latest
```

In [ ]:
def build_index(records, emb, batch_size=200):
    """Embed every chunk and upsert it. Provided."""
    index = get_index(dimension=emb.dimension, metric="cosine")

    vectors = emb.encode([r["text"] for r in records])
    payload = [
        {"id": r["id"], "values": vectors[i].tolist(), "metadata": r["metadata"]}
        for i, r in enumerate(records)
    ]
    # Batched because a single enormous upsert is a good way to hit a request
    # size limit in production. The batch size is not magic; it is small enough
    # to stay well inside the limit at this metadata size.
    for start in range(0, len(payload), batch_size):
        index.upsert(vectors=payload[start:start + batch_size], namespace=L.NAMESPACE)
    return index


with step("Part B: build the baseline index"):
    if not baseline_records:
        raise NotImplementedError("Task B3")
    t0 = time.time()
    baseline_index = build_index(baseline_records, embedder)
    print(baseline_index.describe_index_stats())
    print(f"built in {time.time() - t0:.1f}s")

---

# Part C. Measure it

An index you have not measured is an opinion. This is the part that turns
chunking from a guess into a decision.

`recall_at_k`, `reciprocal_rank`, and `evaluate` are provided. Implementing MRR
is not the lesson.

- **recall@k** asks "did we find it at all". For retrieval augmented generation
  this is the one that matters most. If the right chunk is not in the top k,
  nothing downstream can recover it.
- **MRR** asks "was it near the top".

### Task C1

Write `make_search(index, emb, flt=None)`. It returns a function
`search(query_text, top_k)` that `evaluate` can drive.

Requirements:

- Embed the query with **the same embedder used for the corpus**. This is the
  single most common silent bug in retrieval. Different models produce
  plausible-looking garbage with real-looking scores and no error at all.
- Ask the index for more than `top_k`. Several chunks of the same document will
  come back, and `evaluate` deduplicates up to the document before scoring. If
  you ask for exactly 5 chunks you may end up with only 2 distinct documents.
  Request `top_k * 4`.
- Return a list of dicts, each with at least `chunk_id`, `document_id`, `score`,
  and `text`.
- Pass `filter=flt` and `include_metadata=True` through to the query.

Note that every Pinecone v9 data plane method is **keyword only**. There are no
positional arguments. `index.query(vec, 5)` is a `TypeError`.

**Worked target output.** For the first labeled query at the baseline index:

```
q01  why does the smart thermostat lose wifi after a power cut
   1  0.5670  ts_thermostat_wifi_v1#0      Thermostat drops off the network after a power o
   2  0.4850  ts_thermostat_wifi#0         Thermostat drops off the network after a power o
   3  0.3121  man_thermostat_t40#1         into a provisioning state and does not automatic
```

Your exact scores will match on the offline backend and will differ on a live
embedder. The ordering is what you are looking at, and it is worth a second
look. The top hit is `ts_thermostat_wifi_v1`, the superseded legacy article for
a discontinued product, which tells the technician to replace a working unit
under warranty. It beat the correct article. Hold onto that. You will fix it in
Part F.

In [ ]:
def make_search(index, emb, flt=None):
    """
    Build a search function bound to one index, one embedder, one filter.

    Returns:
        search(query_text, top_k=5) -> list of dicts with keys
        chunk_id, document_id, score, text
    """
    def search(query_text, top_k=K):
        # Same embedder as ingest. Not a similar one. The same object.
        query_vector = emb.encode([query_text])[0]

        response = index.query(
            # Over-fetch: several chunks of one document will come back and get
            # deduplicated to a single document downstream.
            top_k=top_k * 4,
            vector=query_vector.tolist(),
            namespace=L.NAMESPACE,
            filter=flt,
            include_metadata=True,
        )
        return [
            {
                "chunk_id": m.id,
                "document_id": m.metadata["document_id"],
                "score": m.score,
                "text": m.metadata.get("text", ""),
            }
            for m in response.matches
        ]

    return search

In [ ]:
with step("Part C: run one query"):
    if baseline_index is None:
        raise NotImplementedError("Task B3")
    baseline_search = make_search(baseline_index, embedder)
    q = LABELED_QUERIES[0]
    print(f"{q['query_id']}  {q['text']}")
    for rank, hit in enumerate(baseline_search(q["text"], top_k=K)[:3], start=1):
        print(f"  {rank:2d}  {hit['score']:.4f}  {hit['chunk_id']:28s} "
              f"{hit['text'][:48]}")

check("C1: search returns results",
      lambda: len(baseline_search(LABELED_QUERIES[0]["text"], top_k=K)) > 0)
check("C1: results carry document_id",
      lambda: "document_id" in baseline_search(LABELED_QUERIES[0]["text"], top_k=K)[0])
check("C1: over-fetches beyond top_k",
      lambda: len(baseline_search(LABELED_QUERIES[0]["text"], top_k=K)) > K,
      "ask the index for more than top_k so dedup has room")
check("C1: scores are sorted descending",
      lambda: (lambda s: all(s[i]["score"] >= s[i+1]["score"] for i in range(len(s)-1)))(
          baseline_search(LABELED_QUERIES[0]["text"], top_k=K)))

### Task C2

Record the baseline. Call `evaluate` with the labeled query set and your search
function, and store the result in `results["B baseline fixed 1024/0"]`.

**Worked target output.**

```
B baseline fixed 1024/0    recall@5=0.700  MRR=0.629
```

In [ ]:
results["B baseline fixed 1024/0"] = evaluate(
    LABELED_QUERIES, baseline_search, k=K
)
print(f"B baseline fixed 1024/0    "
      f"recall@{K}={results['B baseline fixed 1024/0'][f'recall@{K}']:.3f}  "
      f"MRR={results['B baseline fixed 1024/0']['mrr']:.3f}")

In [ ]:
check("C2: baseline recorded",
      lambda: "B baseline fixed 1024/0" in results)
check("C2: baseline recall@5 is between 0.700 and 0.800, inclusive",
      lambda: 0.700 <= results["B baseline fixed 1024/0"][f"recall@{K}"] <= 0.800)
check("C2: baseline MRR is between 0.625 and 0.725, inclusive",
      lambda: 0.625 <= results["B baseline fixed 1024/0"]["mrr"] <= 0.725)

Roughly seven in ten. Not broken, not good. Support engineers were right, and
now you can say by how much.

Before you tune anything, one detour that will save you more grief than any
amount of parameter sweeping.

---

# Part D. The bug your metric cannot see on its own

The Module 02 slide on chunking strategies showed a sliding window chunker and
asked you to spot a defect. Here it is, exactly as printed, before the
correction. Read it, then measure it.

In [ ]:
def chunk_sliding_v1(tokens, size=512, stride=256):
    """Sliding window chunker, as printed on the slide. Provided, unmodified."""
    chunks = []
    for i in range(0, len(tokens) - size, stride):
        chunks.append(tokens[i:i + size])
    return chunks


sliding_records, sliding_index = [], None
with step("Part D: measure the slide version"):
    sliding_records = chunk_corpus(docs, "sliding_v1", size=512, stride=256)
    sliding_index = build_index(sliding_records, embedder)
    results["D sliding v1 512/256"] = evaluate(
        LABELED_QUERIES, make_search(sliding_index, embedder), k=K)

    base = results["B baseline fixed 1024/0"]
    print(f"records:  {len(sliding_records)}")
    print(f"recall@{K}: {results['D sliding v1 512/256'][f'recall@{K}']:.3f}   "
          f"(baseline {base[f'recall@{K}']:.3f})")
    print(f"MRR:      {results['D sliding v1 512/256']['mrr']:.3f}   "
          f"(baseline {base['mrr']:.3f})")

**Look at those two numbers before you read on.**

Recall fell. MRR went **up**, and by a lot.

If you were watching a ranking dashboard you would have shipped this. That is
the trap. MRR only scores queries by where the first correct answer landed. It
has nothing to say about answers that are not in the index at all.

Something is very wrong and neither metric is going to tell you what. Write the
audit that will.

### Task D1

Write `index_coverage(docs, records)`.

It answers a question no retrieval metric asks: **did every document actually
make it into the index?**

Return a dict with keys `documents_total`, `documents_indexed`,
`documents_missing`, and `missing_ids` (sorted).

**Worked target output**, for the baseline records from Part B:

```python
{'documents_total': 202,
 'documents_indexed': 202,
 'documents_missing': 0,
 'missing_ids': []}
```

In [ ]:
def index_coverage(docs, records):
    """
    Which documents produced no chunks at all?

    Returns:
        dict with documents_total, documents_indexed, documents_missing,
        missing_ids (sorted list)
    """
    # Which document_ids actually appear in the records we are about to index.
    covered = {r["metadata"]["document_id"] for r in records}
    missing = sorted(d["document_id"] for d in docs if d["document_id"] not in covered)
    return {
        "documents_total": len(docs),
        "documents_indexed": len(covered),
        "documents_missing": len(missing),
        "missing_ids": missing,
    }

In [ ]:
from collections import Counter

with step("Part D: coverage audit"):
    cov_baseline = index_coverage(docs, baseline_records)
    cov_sliding = index_coverage(docs, sliding_records)

    print("baseline  ", {k: v for k, v in cov_baseline.items() if k != "missing_ids"})
    print("sliding v1", {k: v for k, v in cov_sliding.items() if k != "missing_ids"})

    missing = set(cov_sliding["missing_ids"])
    print("\nmissing by doc_type:",
          dict(Counter(d["doc_type"] for d in docs if d["document_id"] in missing)))
    print("longest missing document:",
          max((count_tokens(d["text"]) for d in docs if d["document_id"] in missing),
              default=0), "tokens")

check("D1: baseline coverage is complete",
      lambda: index_coverage(docs, baseline_records)["documents_missing"] == 0)
check("D1: sliding v1 indexes only 6 documents",
      lambda: index_coverage(docs, sliding_records)["documents_indexed"] == 6)
check("D1: sliding v1 loses 196 documents",
      lambda: index_coverage(docs, sliding_records)["documents_missing"] == 196)
check("D1: missing_ids is sorted",
      lambda: (lambda m: m == sorted(m))(index_coverage(docs, sliding_records)["missing_ids"]))

**Six documents out of 202.**

Every FAQ entry and every troubleshooting article vanished. The only survivors
are the six installation manuals, which are the only documents longer than 512
tokens.

Go back to `range(0, len(tokens) - size, stride)`. For a 314 token document with
`size=512`, that is `range(0, -198, 256)`, which is empty. The function returns
an empty list, `chunk_corpus` loops zero times, and the document silently ceases
to exist. No exception. No warning. Nothing in any log.

And now Part A pays off: you measured that 97 percent of this corpus is under
512 tokens. That fraction and this failure are the same number.

This also explains the MRR rise. The only documents left in the index were the
long manuals, which happen to rank well when they are relevant. Deleting most of
your corpus improved the average rank of what remained.

> **The transferable lesson.** Latency and error rate tell you the service is
> up. Only a coverage check plus a scheduled recall run tells you the index is
> still good. Wire both into CI.

### Task D2

Write `chunk_sliding_v2(tokens, size, stride)`, the corrected version.

Two fixes:

- A document shorter than or equal to `size` yields exactly one chunk.
- The tail is kept. Stop once a window reaches the end, so you do not emit a
  run of ever shorter trailing chunks.

**Worked target output.**

```
chunk_sliding_v2([], 512, 256)                    -> []
len(chunk_sliding_v2(list(range(400)), 512, 256)) -> 1
len(chunk_sliding_v2(list(range(1000)), 512, 256))-> 3
chunk_sliding_v2(list(range(1000)), 512, 256)[-1][-1] -> 999
```

That last one is the tail check. The final chunk must contain the final token.

In [ ]:
def chunk_sliding_v2(tokens, size=512, stride=256):
    """
    Corrected sliding window.

    - Empty input yields [].
    - A document of length <= size yields exactly one chunk.
    - The final token of the input appears in the final chunk.
    """
    if not tokens:
        return []
    # The fix that matters: a short document is one chunk, not zero.
    if len(tokens) <= size:
        return [tokens]

    chunks = []
    for i in range(0, len(tokens), stride):
        chunks.append(tokens[i:i + size])
        # Tail captured, so stop rather than emitting shrinking trailing windows.
        if i + size >= len(tokens):
            break
    return chunks

In [ ]:
check("D2: empty input yields no chunks",
      lambda: chunk_sliding_v2([], 512, 256) == [])
check("D2: short document yields exactly one chunk",
      lambda: len(chunk_sliding_v2(list(range(400)), 512, 256)) == 1)
check("D2: long document yields several chunks",
      lambda: len(chunk_sliding_v2(list(range(1000)), 512, 256)) == 3)
check("D2: the tail is kept",
      lambda: chunk_sliding_v2(list(range(1000)), 512, 256)[-1][-1] == 999)

with step("Part D: rebuild with the corrected chunker"):
    fixed_records = chunk_corpus(docs, "sliding_v2", size=512, stride=256)
    fixed_index = build_index(fixed_records, embedder)
    results["D sliding v2 512/256"] = evaluate(
        LABELED_QUERIES, make_search(fixed_index, embedder), k=K)

    cov_fixed = index_coverage(docs, fixed_records)
    print(f"\nrecords {len(fixed_records)}   documents indexed "
          f"{cov_fixed['documents_indexed']}/{cov_fixed['documents_total']}")
    print(f"recall@{K}={results['D sliding v2 512/256'][f'recall@{K}']:.3f}  "
          f"MRR={results['D sliding v2 512/256']['mrr']:.3f}")

check("D2: corrected chunker restores full coverage",
      lambda: index_coverage(docs, chunk_corpus(docs, "sliding_v2", size=512, stride=256))
              ["documents_missing"] == 0)
check("D2: corrected recall beats the baseline",
      lambda: (results["D sliding v2 512/256"][f"recall@{K}"]
               > results["B baseline fixed 1024/0"][f"recall@{K}"]))

---

# Part E. Now tune it, on evidence

Coverage is fixed. Every document is in the index. Only now is it worth arguing
about chunk size.

One more thing to change first. Both chunkers so far cut on raw token count,
which means they cut mid-sentence and often mid-word-sequence. The module is
explicit: prefer natural boundaries. Splitting mid-sentence degrades embedding
quality measurably, because the vector for half a thought is a vector for
nothing in particular.

### Task E1

Write `chunk_paragraphs(text, size, overlap)`.

Pack whole paragraphs into chunks up to `size` tokens. Never split a paragraph.
Carry trailing paragraphs forward as overlap.

Algorithm:

1. `split_paragraphs(text)` gives you the paragraph list. It is already imported.
2. Accumulate paragraphs into a current chunk, tracking the running token count.
3. When adding the next paragraph would exceed `size` **and** the current chunk
   is not empty, emit the current chunk.
4. After emitting, seed the next chunk with trailing paragraphs of the one you
   just emitted, taking them from the end while the running total stays within
   `overlap`.
5. Emit whatever is left at the end.

A paragraph longer than `size` on its own still becomes one chunk. That is
correct. Splitting it would be exactly the mid-sentence cut you are avoiding.

**Worked target output.**

```
chunk_paragraphs("", 384, 64)                              -> []
len(chunk_paragraphs(docs_by_id["faq_returns"]["text"], 384, 64))       -> 1
len(chunk_paragraphs(docs_by_id["man_thermostat_t40"]["text"], 384, 64))-> 12
```

And over the corpus:

```
chunk_corpus(docs, "paragraph", size=384, overlap=64)  ->  257 records
```

In [ ]:
def chunk_paragraphs(text, size=384, overlap=64):
    """
    Pack whole paragraphs into chunks of at most `size` tokens.

    Never splits a paragraph. Carries trailing paragraphs forward as overlap.
    Returns a list of strings, paragraphs rejoined with a blank line.
    """
    paragraphs = split_paragraphs(text)
    if not paragraphs:
        return []

    chunks = []
    current, current_tokens = [], 0

    for para in paragraphs:
        n = count_tokens(para)

        # Emit when this paragraph would overflow. The "and current" guard means
        # an oversized single paragraph still becomes its own chunk rather than
        # emitting an empty one and looping.
        if current_tokens and current_tokens + n > size:
            chunks.append("\n\n".join(current))

            # Seed the next chunk with trailing paragraphs, newest first, while
            # they fit inside the overlap budget.
            carry, carry_tokens = [], 0
            for prev in reversed(current):
                prev_n = count_tokens(prev)
                if carry_tokens + prev_n > overlap:
                    break
                carry.insert(0, prev)
                carry_tokens += prev_n
            current, current_tokens = list(carry), carry_tokens

        current.append(para)
        current_tokens += n

    if current:
        chunks.append("\n\n".join(current))
    return chunks

In [ ]:
docs_by_id = {d["document_id"]: d for d in docs}

check("E1: empty text yields no chunks",
      lambda: chunk_paragraphs("", 384, 64) == [])
check("E1: a short FAQ stays a single chunk",
      lambda: len(chunk_paragraphs(docs_by_id["faq_returns"]["text"], 384, 64)) == 1)
check("E1: a long manual splits into 12 chunks",
      lambda: len(chunk_paragraphs(docs_by_id["man_thermostat_t40"]["text"], 384, 64)) == 12)
check("E1: no chunk splits a paragraph",
      lambda: all(
          p in docs_by_id["man_thermostat_t40"]["text"]
          for c in chunk_paragraphs(docs_by_id["man_thermostat_t40"]["text"], 384, 64)
          for p in c.split("\n\n")))
check("E1: corpus yields 257 records at 384/64",
      lambda: len(chunk_corpus(docs, "paragraph", size=384, overlap=64)) == 257)

### The sweep

Provided. This is the measurement that replaces the argument.

In [ ]:
SWEEP = [
    ("E paragraph 512/64", dict(size=512, overlap=64)),
    ("E paragraph 384/64", dict(size=384, overlap=64)),
    ("E paragraph 256/32", dict(size=256, overlap=32)),
    ("E paragraph 192/24", dict(size=192, overlap=24)),
]

with step("Part E: sweep chunk sizes"):
    for label, kw in SWEEP:
        recs = chunk_corpus(docs, "paragraph", **kw)
        idx = build_index(recs, embedder)
        indexes[label] = (idx, recs)
        results[label] = evaluate(LABELED_QUERIES, make_search(idx, embedder), k=K)
        cov = index_coverage(docs, recs)
        print(f"{label:22s} chunks={len(recs):4d}  "
              f"coverage={cov['documents_indexed']}/{cov['documents_total']}  "
              f"recall@{K}={results[label][f'recall@{K}']:.3f}  "
              f"MRR={results[label]['mrr']:.3f}")

if results:
    fig, ax = plot_metric_comparison(results, k=K)
    plt.show()

In [ ]:
check("E: 384/64 recall@5 is between 0.700 and 0.800, inclusive",
      lambda: 0.700 <= results["E paragraph 384/64"][f"recall@{K}"] <= 0.800)
check("E: 192/24 recall@5 is between 0.750 and 0.950, inclusive",
      lambda: 0.750 <= results["E paragraph 192/24"][f"recall@{K}"] <= 0.950)
check("E: smaller chunks beat the 1024 baseline on recall",
      lambda: (results["E paragraph 192/24"][f"recall@{K}"]
               > results["B baseline fixed 1024/0"][f"recall@{K}"]))
check("E: every sweep configuration keeps full coverage",
      lambda: len(indexes) == len(SWEEP) and
              all(index_coverage(docs, recs)["documents_missing"] == 0
                  for _idx, recs in indexes.values()))

**Read the sweep honestly.** Recall climbs the whole way down to 192 tokens. MRR
does not: it wobbles and is no better at 192 than at the baseline.

Two things are happening at once. Smaller chunks isolate the answer, which helps
recall. Smaller chunks also give 200 distractor documents more small, sharply
worded chunks that can outrank the right answer, which hurts rank quality.

And the metric has a blind spot you have to supply yourself. Recall cannot see
that a 192 token chunk may be too small to be *useful* to the model that reads
it next. "Set the value to 3" retrieves beautifully and tells the reader
nothing. You are optimising a proxy. Know where the proxy stops tracking the
thing you actually care about.

A defensible landing spot on this corpus is **384 with 64 overlap**: most of the
recall gain, no MRR regression, chunks still large enough to carry a whole
procedure. Your reasoning matters more than the number.

---

# Part F. Metadata is not decoration

Three documents in this corpus are superseded revisions covering discontinued
products. They are marked `is_active: False`. They are also, unfortunately,
extremely good semantic matches, because a legacy article about a thermostat
losing wifi looks almost exactly like the current one.

A stale document is worse than a missing one. A missing document produces "I do
not know". A stale document produces a confident wrong answer with a citation.

In [ ]:
stale = [d for d in docs if not d["is_active"]]
print(f"{len(stale)} superseded documents in the corpus:")
for d in stale:
    print(f"  {d['document_id']:28s} {d['source']}")

stale_ids = {d["document_id"] for d in stale}
leaks = []
with step("Part F: count stale leakage"):
    best_index, best_records = indexes["E paragraph 384/64"]
    unfiltered_search = make_search(best_index, embedder)

    for q in LABELED_QUERIES:
        seen, ordered = set(), []
        for hit in unfiltered_search(q["text"], top_k=K):
            if hit["document_id"] not in seen:
                seen.add(hit["document_id"])
                ordered.append(hit["document_id"])
        if set(ordered[:K]) & stale_ids:
            leaks.append(q["query_id"])

    print(f"\nqueries returning a superseded document in the top {K}: "
          f"{len(leaks)} of {len(LABELED_QUERIES)}")
    print(f"  {leaks}")

### Task F2

Build two filters.

`ACTIVE_ONLY` keeps only records where `is_active` is `True`.

`ACTIVE_AND_RECENT` additionally requires `updated_at` to be at or after
midnight UTC on 1 January 2025. Use `to_epoch` to produce the operand.

Filter syntax is a dict of field to operator dict, for example
`{"doc_type": {"$eq": "faq"}}`. Operators used here are `$eq` and `$gte`.

**Worked target output.**

```python
ACTIVE_ONLY        -> {'is_active': {'$eq': True}}
ACTIVE_AND_RECENT  -> {'is_active': {'$eq': True},
                       'updated_at': {'$gte': 1735689600}}
```

The exact epoch integer depends on your machine time zone, which is itself worth
noticing. Compute it, do not type it.

In [ ]:
ACTIVE_ONLY = {"is_active": {"$eq": True}}

ACTIVE_AND_RECENT = {
    "is_active": {"$eq": True},
    # Epoch integer. A string here matches nothing and raises nothing.
    "updated_at": {"$gte": to_epoch("2025-01-01T00:00:00Z")},
}

print("ACTIVE_ONLY      ", ACTIVE_ONLY)
print("ACTIVE_AND_RECENT", ACTIVE_AND_RECENT)

In [ ]:
leaks_after = None
with step("Part F: evaluate with filters"):
    if ACTIVE_ONLY is None or ACTIVE_AND_RECENT is None:
        raise NotImplementedError("Task F2")
    results["F active filter"] = evaluate(
        LABELED_QUERIES, make_search(best_index, embedder, ACTIVE_ONLY), k=K)
    results["F active + recent"] = evaluate(
        LABELED_QUERIES, make_search(best_index, embedder, ACTIVE_AND_RECENT), k=K)

    for label in ("E paragraph 384/64", "F active filter", "F active + recent"):
        print(f"{label:22s} recall@{K}={results[label][f'recall@{K}']:.3f}  "
              f"MRR={results[label]['mrr']:.3f}")

    leaks_after = []
    filtered_search = make_search(best_index, embedder, ACTIVE_ONLY)
    for q in LABELED_QUERIES:
        seen, ordered = set(), []
        for hit in filtered_search(q["text"], top_k=K):
            if hit["document_id"] not in seen:
                seen.add(hit["document_id"]); ordered.append(hit["document_id"])
        if set(ordered[:K]) & stale_ids:
            leaks_after.append(q["query_id"])
    print(f"\nsuperseded documents leaking after the filter: {len(leaks_after)}")

check("F2: ACTIVE_ONLY filters on is_active",
      lambda: ACTIVE_ONLY == {"is_active": {"$eq": True}})
check("F2: ACTIVE_AND_RECENT uses an epoch integer",
      lambda: isinstance(ACTIVE_AND_RECENT["updated_at"]["$gte"], int))
check("F2: the filter removes every superseded document",
      lambda: len(leaks_after) == 0)
check("F2: MRR improves once stale content is excluded",
      lambda: results["F active filter"]["mrr"] > results["E paragraph 384/64"]["mrr"])

### The silent failure, on purpose

Run this. It is the single most valuable cell in Part F.

In [ ]:
BROKEN = {"updated_at": {"$gte": "2025-01-01T00:00:00Z"}}   # a string, not an int
broken_result = None
with step("Part F: the silent failure"):
    broken_result = evaluate(
        LABELED_QUERIES, make_search(best_index, embedder, BROKEN), k=K)
    print(f"string operand on a numeric field: "
          f"recall@{K}={broken_result[f'recall@{K}']:.3f}  "
          f"MRR={broken_result['mrr']:.3f}")
    print("no exception was raised")

check("F: a string date silently matches nothing",
      lambda: broken_result[f"recall@{K}"] == 0.0)

Zero. Not an error, not a warning, not a log line. Every query returned nothing
and the code path looks completely healthy.

In production this arrives as a support ticket saying search stopped working,
three weeks after a well-meaning refactor changed a timestamp to an ISO string
because it read better in the logs.

### Task F3

Write `delete_document(index, document_id, namespace)`.

This is where ID design stops being cosmetic. Because chunk IDs are
`{document_id}#{chunk_index}`, deleting a document is two calls: list the IDs
sharing the prefix, then delete that list.

`index.list(prefix=..., namespace=...)` returns an **iterator of pages**.
Iterating a page yields `ListItem` objects, not strings. You want `item.id`.
This changed in SDK v9 and is a common upgrade break.

Return the list of deleted IDs.

**Worked target output.**

```
delete_document(index, "man_thermostat_t40")  -> 12 ids deleted
vectors 257 -> 245
```

Note also what you **cannot** rely on. Pinecone cloud added delete by metadata
filter in late 2025, but Pinecone Local pins API version `2025-01`, which
predates it. Prefix deletion works everywhere and is cheaper. Design for it.

In [ ]:
def delete_document(index, document_id, namespace=None):
    """
    Delete every chunk of one document by ID prefix.

    Returns:
        the sorted list of chunk IDs that were deleted
    """
    namespace = L.NAMESPACE if namespace is None else namespace

    # list() yields PAGES. Iterating a page yields ListItem objects in SDK v9,
    # so take .id rather than using the item directly.
    ids = sorted(
        item.id
        for page in index.list(prefix=f"{document_id}#", namespace=namespace)
        for item in page
    )
    if ids:
        index.delete(ids=ids, namespace=namespace)
    return ids

In [ ]:
scratch_index, deleted, before, after = None, [], 0, 0
with step("Part F: prefix delete"):
    scratch_index = build_index(best_records, embedder)
    before = scratch_index.describe_index_stats().total_vector_count
    deleted = delete_document(scratch_index, "man_thermostat_t40")
    after = scratch_index.describe_index_stats().total_vector_count
    print(f"deleted {len(deleted)} ids: {deleted[:3]} ...")
    print(f"vectors {before} -> {after}")

check("F3: deletes 12 chunks of the thermostat manual",
      lambda: len(deleted) == 12)
check("F3: every deleted id carries the document prefix",
      lambda: bool(deleted) and
              all(i.startswith("man_thermostat_t40#") for i in deleted))
check("F3: the index shrinks by exactly that many vectors",
      lambda: bool(deleted) and before - after == len(deleted))
check("F3: the document is gone from search",
      lambda: all(h["document_id"] != "man_thermostat_t40"
                  for h in make_search(scratch_index, embedder)(
                      "why does the smart thermostat lose wifi after a power cut", top_k=K)))

---

# Part G. Stretch: test the day's thesis with your own numbers

The claim from the opening slide was that chunking strategy moves retrieval
quality more than embedding model choice does. You now have everything you need
to check it rather than believe it.

`get_embedder("alternate", ...)` returns a genuinely different embedding model
with a **different dimension**. Not the same model resized. Different features,
different geometry, different vector space.

### Task G1

1. Build the alternate embedder.
2. Confirm you cannot mix the two spaces. Encode any string with each and show
   the dimensions differ, so a vector from one cannot be scored against an index
   built from the other.
3. Rebuild the **same** chunk records with the alternate embedder, evaluate, and
   store under `results["G alternate model 384/64"]`.
4. Compute both deltas from the baseline and print which lever moved more.

**Worked target output.**

```
primary   lsa dim=96      alternate lsa dim=48
G alternate model 384/64   recall@5=0.683  MRR=0.527

best chunking configuration: E paragraph 192/24
chunking delta (baseline -> E paragraph 192/24  ): recall +0.117
model    delta (384/64 primary -> alternate) : recall -0.067

larger lever: chunking
```

In [ ]:
alt_embedder = get_embedder("alternate", dim=EMBED_DIM, background=background)
print(f"primary   {embedder.name} dim={embedder.dimension}      "
      f"alternate {alt_embedder.name} dim={alt_embedder.dimension}")

# Spaces are not interchangeable. Different dimension is the loud version of the
# problem. The quiet version is two models of the SAME dimension, which produces
# real-looking scores and meaningless results with no error at all.
probe = "thermostat loses wifi after a power cut"
assert embedder.encode([probe]).shape[1] != alt_embedder.encode([probe]).shape[1]

# The alternate model emits wider vectors than the primary (e.g. 768 vs 384 on a
# live sentence-transformers backend). At the default batch size that doubles the
# upsert payload and can trip Pinecone Local's request-size limit ([413]), so cut
# the batch size in proportion to the wider vectors.
alt_batch = max(1, (200 * embedder.dimension) // alt_embedder.dimension)
alt_index = build_index(best_records, alt_embedder, batch_size=alt_batch)
results["G alternate model 384/64"] = evaluate(
    LABELED_QUERIES, make_search(alt_index, alt_embedder), k=K)
print(f"G alternate model 384/64   "
      f"recall@{K}={results['G alternate model 384/64'][f'recall@{K}']:.3f}  "
      f"MRR={results['G alternate model 384/64']['mrr']:.3f}")

# The chunking lever is the whole range you explored, baseline to best, not one
# arbitrary point inside it. The model lever is the swap at fixed chunking.
sweep_labels = [label for label, _ in SWEEP]
best_label = max(sweep_labels, key=lambda lab: results[lab][f"recall@{K}"])

chunking_delta = (results[best_label][f"recall@{K}"]
                  - results["B baseline fixed 1024/0"][f"recall@{K}"])
model_delta = (results["G alternate model 384/64"][f"recall@{K}"]
               - results["E paragraph 384/64"][f"recall@{K}"])

print(f"\nbest chunking configuration: {best_label}")
print(f"chunking delta (baseline -> {best_label:<20s}): recall {chunking_delta:+.3f}")
print(f"model    delta (384/64 primary -> alternate) : recall {model_delta:+.3f}")
print(f"\nlarger lever: "
      f"{'chunking' if abs(chunking_delta) >= abs(model_delta) else 'model choice'}")

In [ ]:
def mismatch_is_caught():
    """Query an index built on one model using a vector from the other one."""
    # On Pinecone Local there is a single shared index and build_index recreates
    # it on every call, so the earlier best_index handle now points at the
    # 768-dim alternate index. Rebuild the primary (384-dim) index so the probe
    # queries a genuinely smaller space than the alternate vector; otherwise the
    # dimensions happen to match and nothing is raised.
    primary_index = build_index(best_records, embedder)
    try:
        wrong_vector = alt_embedder.encode(["thermostat wifi"])[0].tolist()
        primary_index.query(top_k=3, vector=wrong_vector, namespace=L.NAMESPACE)
        # Reaching here means the call succeeded, which is the dangerous case.
        print("no error raised, results would be meaningless")
        return False
    except Exception as exc:
        print(f"caught as expected: {type(exc).__name__}: {str(exc)[:110]}")
        return True


caught = None
with step("Part G: dimension mismatch probe"):
    if alt_embedder is None:
        raise NotImplementedError("Task G1")
    caught = mismatch_is_caught()

check("G1: alternate model evaluated",
      lambda: "G alternate model 384/64" in results)
check("G1: the two embedding spaces have different dimensions",
      lambda: embedder.dimension != alt_embedder.dimension)
check("G1: a dimension mismatch is rejected rather than silently accepted",
      lambda: caught is True)

**What the numbers say on this corpus.** The chunking change helped, by 0.117
recall across the range you explored. The model change hurt, by 0.067. Chunking
was the larger lever and it was the only one that moved in the direction you
wanted.

Be precise about what is being compared. The chunking lever is the whole span
from the 1024 baseline to the best configuration you found, because that is the
range genuinely available to you. Comparing the model swap against a single
mid-range chunking point would flatter the model swap and would not reflect a
decision anyone actually faces.

Be careful about how far you generalise. One corpus, one query set, thirty
labels, two models. What this result actually licenses is a claim about
sequencing: measure chunking first, because it is cheap to change and it moved
the metric here. It does not license "embedding models do not matter".

**The dimension mismatch is the good news case.** It raised. The dangerous
version is two different models that happen to share a dimension. Then every
call succeeds, every score looks plausible, and every result is meaningless.
Nothing in the stack will tell you. The only defence is discipline: the query
embedding client must be the same object as the ingest embedding client.

---

# Part H. Stretch: refresh the index without rebuilding it

Treat the index like a cache that needs invalidation. Documents change, products
are discontinued, procedures are updated, and the index does not know.

A full re-embed is simple, expensive, and safe. Incremental refresh is what you
run nightly.

### Task H1

Write `incremental_refresh(documents, existing)`.

`existing` maps `document_id` to `{"hash": str, "chunk_ids": [str, ...]}`,
the state from the last indexing run.

Return `(to_reembed, chunk_ids_to_delete, new_docs)`:

- **changed**: the ID is known and the content hash differs. Re-embed it, and
  delete its old chunk IDs so stale chunks do not linger alongside new ones.
- **new**: the ID is not in `existing`. Embed it. Nothing to delete.
- **removed**: the ID is in `existing` but not in `documents`. Delete its chunk
  IDs. It is not re-embedded, because it no longer exists upstream.

Use `content_hash(text)`, which is SHA-256. Do **not** use Python's built-in
`hash()`. It is salted per process, so it returns a different value on every
interpreter restart and would mark your entire corpus as changed every night.

**Worked target output**, against a corpus with one edit, one deletion, and one
addition:

```
changed docs to re-embed: 1  ['faq_airsense_a2']
new docs:                 1  ['faq_new_policy']
chunk ids to delete:      2
full re-embed: 257 chunks   incremental: 2 documents
```

In [ ]:
def incremental_refresh(documents, existing):
    """
    Decide what changed since the last indexing run.

    Args:
        documents: the current corpus
        existing:  {document_id: {"hash": str, "chunk_ids": [str, ...]}}

    Returns:
        (to_reembed, chunk_ids_to_delete, new_docs)
    """
    to_reembed, chunk_ids_to_delete, new_docs = [], [], []

    for doc in documents:
        # SHA-256, stable across processes. Python's hash() is salted per
        # process and would report the whole corpus as changed every night.
        digest = content_hash(doc["text"])
        doc_id = doc["document_id"]

        if doc_id in existing:
            if existing[doc_id]["hash"] != digest:
                to_reembed.append(doc)
                # Drop the old chunks. Otherwise stale chunks survive next to
                # the new ones and both are retrievable.
                chunk_ids_to_delete.extend(existing[doc_id]["chunk_ids"])
        else:
            new_docs.append(doc)

    # Documents that disappeared upstream. Delete, do not re-embed.
    current = {d["document_id"] for d in documents}
    for gone in sorted(set(existing) - current):
        chunk_ids_to_delete.extend(existing[gone]["chunk_ids"])

    return to_reembed, chunk_ids_to_delete, new_docs

In [ ]:
# Reconstruct last night's state from the records we already indexed.
existing_state = {}
for r in best_records:
    doc_id = r["metadata"]["document_id"]
    existing_state.setdefault(doc_id, {"hash": None, "chunk_ids": []})["chunk_ids"].append(r["id"])
for d in docs:
    if d["document_id"] in existing_state:
        existing_state[d["document_id"]]["hash"] = content_hash(d["text"])

# Simulate one night of upstream change: one edit, one deletion, one addition.
changed_corpus = [dict(d) for d in docs]
changed_corpus[0]["text"] = changed_corpus[0]["text"] + "\n\nNew section added today."
edited_id = changed_corpus[0]["document_id"]
removed_id = changed_corpus[5]["document_id"]
changed_corpus = [d for d in changed_corpus if d["document_id"] != removed_id]
changed_corpus.append({
    "document_id": "faq_new_policy", "source": "faq_new_policy.md",
    "doc_type": "faq", "product_line": "policy",
    "created_at": "2026-06-01T08:00:00Z", "updated_at": "2026-06-01T08:00:00Z",
    "is_active": True, "text": "A brand new policy article."})

to_reembed, to_delete, new_docs = [], [], []
with step("Part H: plan the refresh"):
    to_reembed, to_delete, new_docs = incremental_refresh(changed_corpus, existing_state)
print(f"edited upstream:  {edited_id}")
print(f"removed upstream: {removed_id}")
print(f"changed docs to re-embed: {len(to_reembed)}  {[d['document_id'] for d in to_reembed]}")
print(f"new docs:                 {len(new_docs)}  {[d['document_id'] for d in new_docs]}")
print(f"chunk ids to delete:      {len(to_delete)}")
print(f"\nfull re-embed: {len(best_records)} chunks   "
      f"incremental: {len(to_reembed) + len(new_docs)} documents")

check("H1: detects exactly the edited document",
      lambda: [d["document_id"] for d in to_reembed] == [edited_id])
check("H1: detects the new document",
      lambda: [d["document_id"] for d in new_docs] == ["faq_new_policy"])
check("H1: schedules deletes for edited and removed documents",
      lambda: (set(existing_state[removed_id]["chunk_ids"]) <= set(to_delete)
               and set(existing_state[edited_id]["chunk_ids"]) <= set(to_delete)))
check("H1: does not re-embed the removed document",
      lambda: bool(to_reembed or new_docs) and
              removed_id not in {d["document_id"] for d in to_reembed + new_docs})
check("H1: an unchanged corpus produces no work at all",
      lambda: incremental_refresh(docs, existing_state) == ([], [], []))

---

# Wrap up

In [ ]:
print(f"{'configuration':30s} {'recall@'+str(K):>10s} {'MRR':>8s}")
for label, m in results.items():
    print(f"{label:30s} {m[f'recall@{K}']:10.3f} {m['mrr']:8.3f}")
print()
check_summary()

## What you built that you get to keep

- **The corpus and the chunker.** Point them at any document collection.
- **The labeled query set**, and more importantly the workflow for making one
  from real query logs. Fifty to a hundred representative queries is enough to
  tell you whether a change helped. You are measuring a delta, not certifying an
  absolute.
- **The evaluation harness.** This is the piece that pays for itself, in every
  migration and again in the capstone.
- **The coverage audit.** Wire it into CI next to the recall run.
- **A decision framework.** Coverage first, chunking second, model third, and
  reranking when recall@50 diverges from recall@5.

## The four things worth remembering

1. A retrieval metric cannot see a document that never made it into the index.
   Check coverage separately, always.
2. MRR went up while the index lost 97 percent of the corpus. Never read one
   metric alone.
3. Timestamps are epoch integers. A string operand matches nothing and raises
   nothing.
4. Your chunk ID scheme decides, on day one, whether you can delete a document
   on day four hundred.

## Where this goes next

You now have both halves. Fine-tune for behavior, retrieve for facts. The next
module turns retrieved chunks into a grounded answer, and Week 6 makes the
combined system measurable and reproducible.